In [1]:
from qiskit import QuantumCircuit, execute, Aer,  ClassicalRegister
from copy import deepcopy
from math import sqrt

def get_sso(dist1, dist2):                                                                                                                                                                                                                           
    '''Returns the square of the statistical overlap. 
    dist1 and dist2 are probability distributions.
    dist1: list
    dits2: list'''
    sum=0
    common_keys=dist1.keys() & dist2.keys()
    # print(f"dist1: {dist1}")
    # print(f"dist2: {dist2}")
    for key in common_keys:
        # print(f"key: {key}")
        # print(f"dist1[key]: {dist1[key]}")
        # print(f"dist2[key]: {dist2[key]}")
        sum+=sqrt(dist1[key]*dist2[key])
    return sum**2

def post_select_on_ancilla(res, ancilla_value, new_nqubits):
    """
    strip the results where ancilla was not equal to `ancilla_value`
    This is some voodoo copied from 
    https://qiskit.org/documentation/tutorials/noise/8_tomography.html#2-Qubit-Conditional-State-Tomography  
    """
    assert(isinstance(ancilla_value, str))
    res_new = deepcopy(res)
    for resultidx, _ in enumerate(res.results): 
        old_counts = res.get_counts(resultidx)
        new_counts = {}
        res_new.results[resultidx].header.creg_sizes = [res_new.results[resultidx].header.creg_sizes[1]]
        res_new.results[resultidx].header.clbit_labels = res_new.results[resultidx].header.clbit_labels[0:-1]
        res_new.results[resultidx].header.memory_slots = new_nqubits 
     
        for reg_key in old_counts:
            reg_bits = reg_key.split(' ')
            assert(len(reg_bits) == 2)
            assert(len(reg_bits[1]) == 1)
            if reg_bits[1]==ancilla_value:
                new_counts[reg_bits[0]]=old_counts[reg_key]
     
            res_new.results[resultidx].data.counts = new_counts
    return res_new                                                 

In [2]:
qc1 = QuantumCircuit.from_qasm_file("qubits_5_CNOTS_30_circuit_4_checks.qasm")
creg1 = ClassicalRegister(5)
qc1.add_register(creg1)
qc1.measure([0,1,2,3,4],creg1)

qc2 = QuantumCircuit.from_qasm_file("qubits_5_CNOTS_30_circuit_4_nochecks.qasm")
creg2 = ClassicalRegister(5)
qc2.add_register(creg2)
qc2.measure([0,1,2,3,4],creg2)

In [3]:
for _ in range(20):
    res1 = execute(qc1, Aer.get_backend('qasm_simulator'), shots=50000).result()
    counts2 = execute(qc2, Aer.get_backend('qasm_simulator'), shots=50000).result().get_counts()

    counts = res1.get_counts()

    res1 = post_select_on_ancilla(res1, '0', 5)
    counts1 = res1.get_counts()
    assert set(counts.values()) == set(counts1.values())

    def normalize_counts(counts):
        total_counts = sum(counts.values())
        return {k: v/total_counts for k,v in counts.items()}

    counts1 = normalize_counts(counts1)
    counts2 = normalize_counts(counts2)

    print(get_sso(counts1, counts2))

0.9997243252246178
0.9997005660184297
0.9997439441620416
0.9996471092942784
0.9997502421993514
0.9998399052195568
0.999682439463023
0.9996232161390527
0.9996861889480261
0.9998150999666329
0.9997820104117866
0.9997059765848181
0.9996933576648691
0.9996742435218903
0.9995797281470393
0.9995879360502328
0.9997356349428125
0.9997011084064297
0.9997966627486509
0.9996700589875079


Checks out!

Now let's try with noise

In [4]:
true_counts = normalize_counts(execute(qc2, Aer.get_backend('qasm_simulator'), shots=50000).result().get_counts())

In [5]:
import qiskit.providers.aer.noise as noise

mean_improv = 0
ntries = 50

for _ in range(ntries):
    # Error probabilities
    prob_1 = 0.002  # 1-qubit gate
    prob_2 = 0.02   # 2-qubit gate

    # Depolarizing quantum errors
    error_1 = noise.depolarizing_error(prob_1, 1)
    error_2 = noise.depolarizing_error(prob_2, 2)

    # Add errors to noise model
    noise_model = noise.NoiseModel()
    noise_model.add_all_qubit_quantum_error(error_1, ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(error_2, ['cx'])

    # Get basis gates from noise model
    basis_gates = noise_model.basis_gates

    # Perform a noise simulation
    res1 = execute(qc1, Aer.get_backend('qasm_simulator'),
                     basis_gates=basis_gates,
                     noise_model=noise_model,
                     shots=1000,
                  ).result()
    res1 = post_select_on_ancilla(res1, '0', 5)
    counts1 = normalize_counts(res1.get_counts())

    counts2 = execute(qc2, Aer.get_backend('qasm_simulator'),
                     basis_gates=basis_gates,
                     noise_model=noise_model,
                     shots=1000,
                  ).result().get_counts()
    counts2 = normalize_counts(counts2)

    print('Improvement:', get_sso(true_counts, counts1) - get_sso(true_counts, counts2))
    mean_improv += get_sso(true_counts, counts1) - get_sso(true_counts, counts2)

print('Mean:', mean_improv / ntries)

Improvement: 0.010166768081601774
Improvement: 0.015049734176544649
Improvement: 0.019879407231133128
Improvement: 0.030282065576250905
Improvement: 0.0109707702602716
Improvement: 0.0038142431344503347
Improvement: 0.015879136444328035
Improvement: 0.0008828432984020473
Improvement: 0.020119086400910313
Improvement: 0.015124653944300626
Improvement: 0.015964849557912997
Improvement: 0.013620458833381588
Improvement: 0.005514264627453969
Improvement: 0.013590163019683188
Improvement: 0.005907797980437057
Improvement: 0.030075548393101892
Improvement: 0.029617552155787763
Improvement: 0.02302120101673044
Improvement: 0.008659367465598589
Improvement: -0.00192999965524554
Improvement: 0.022144755570069008
Improvement: 0.010175857070831795
Improvement: 0.004553651375316492
Improvement: 0.013277468838498896
Improvement: -0.0033346496738340115
Improvement: 0.010227364693127683
Improvement: 0.023688448778861115
Improvement: 0.023657127835995606
Improvement: 0.02524410127432708
Improvement: 0